In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt



In [2]:
print(bytes([0x41]))

b'A'


## Tokenization

Given a corpus of text, we want to train a tokenizer, T, to be able to split the text into an ordered list of tokens losslessly. There are two main methods which will be implemented in this notebook:

Method 1: Character-Level Tokenization
- Each character is considered it's own token. This is the simplest method to tokenize text, and results in very long sequences, and subpar performance.

Method 2: BPE
- An algorithm that can tokenize words into subwords effectively. This will be implemented in Rust.


In [3]:


class CharacterTokenizer:

    def __init__(self):
        self.char_to_idx = {}
        self.idx_to_char = {}

        for x in range(256):
            self.char_to_idx[bytes([x])] = x
            self.idx_to_char[x] = bytes([x])

    def tokenize(self, text: bytes):
        if type(text) == str:
            text = text.encode('utf-8')
            
        tokens = torch.tensor([self.char_to_idx[char] for char in text], dtype=torch.long)
        return tokens
    
    def decode(self, tokens):
        text = ''
        for token in tokens:
            text += self.idx_to_char[token]
        return text



## Multi-Head Attention

In transformer neural networks, the multihead attention blocks allow each token to attend to its context. Each multihead attention block takes in a tensor of size (batch, seq_len, d_model), X, representing the embeddings of the sequence of tokens from the previous transformer block (or the original embeddings if it's the first block), and projects them 3 times with learned weight matrices, $W_i^Q,W_i^K,W_i^V$, ($W_i^Q,W_i^K$ have a size of (d_model, d_k) and $W_i^V$ has a size of (d_model, d_v)) to Q, K, and V, respectively for each attention block (h attention blocks in total). An attention block takes in 3 matrices, Q (seq_len, d_k), K (seq_len, d_k), and V (seq_len, d_v) and outputs a matrix of size (seq_len, d_v). After each attention block is calculated, the results from each head are concatenated across the d_v axis (each head is (seq_len, d_v) so h heads concatenated will be (seq_len, d_v * h = d_model)).

#### Why use multiple heads instead of just 1 attention block with d_model dimensional inputs and outputs?

This allows the model to be able to learn multiple distinct notions of "relevance", which would not be possible with a single head, due to averaging.

</br>

$$
\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V
$$

$$
\text{MultiHeadAttention} = \text{concat}(\text{head}_1,\text{head}_2,\text{head}_3,...,\text{head}_h)
$$

$$
\text{head}_i = \text{Attention}(XW_i^Q,XW_i^K,XW_i^V)
$$

## Transformer Decoders
In a decoder block, we mask the future tokens before computing the softmax, like this:


$$
\text{DecoderAttention}(Q,K,V) = \text{softmax}(\frac{QK^T + \text{Mask}}{\sqrt{d_k}})V
$$

where $\text{Mask}_{ij} = \begin{cases}
-\infty & \text{if } j > i \\
0 & \text{otherwise}
\end{cases}$

</br>

Example (if seq_len = 4):
</br>
$\text{Mask} = \begin{bmatrix}
    0 & -\infty & -\infty & -\infty \\
    0 & 0 & -\infty & -\infty \\
    0 & 0 & 0 & -\infty \\
    0 & 0 & 0 & 0 \\
\end{bmatrix}$

Notes:
- For implementations of very large models, the attention is computed in a different way to reduce redundant computation.

- Instead of multi-head attention, most llms use multi-query attention, where $W_i^K$ and $W_i^V$ is shared between heads--only the weight matrices for queries are different between heads.

In [8]:

class Attention(nn.Module):
    def __init__(self,d_k,d_v):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.scale = np.sqrt(d_k)

    def forward(self,Q,K,V):
        x = torch.matmul(Q, torch.transpose(K,-2,-1)) / self.scale

        # softmax over each row (a row, i, represents the "relevance" that each token, j, in the sequence has with token i)
        # creates the attention matrix
        x = F.softmax(x, -2)

        # for each token in position i, the new token in position i, after applying attention is the weighted sum of all tokens, j, using the weights from row i of the attention matrix
        x = torch.matmul(x, V)
        return x

class DecoderAttention(nn.Module):
    def __init__(self,d_k,d_v):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.scale = np.sqrt(d_k)

    def forward(self,Q,K,V):
        seq_len = Q.shape[-2]

        # torch.full creates a matrix of size n x n filled with a given element. torch.triu takes in a matrix and returns a matrix of the same size with all but elements above the main diagonal set to zero.
        # creates the attention mask that prevents future tokens from influencing past tokens
        mask = torch.triu(torch.full((seq_len, seq_len), -np.inf))

        x = (torch.matmul(Q, torch.transpose(K,-2,-1)) + mask) / self.scale

        # softmax over each row (a row, i, represents the "relevance" that each token, j, in the sequence has with token i)
        # creates the attention matrix
        x = F.softmax(x, -2)

        # for each token in position i, the new token in position i, after applying attention is the weighted sum of all tokens, j, using the weights from row i of the attention matrix
        x = torch.matmul(x, V)
        return x


class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,d_k,d_v,heads=8):
        super().__init__()

        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.h = heads

        self.attention = Attention(d_k * self.h, d_v * self.h)

        # input size should be the last dimension of the tensor, which is d_model
        self.query_weights = nn.Linear(in_features=d_model, out_features=d_k * self.h, bias=False)
        self.key_weights = nn.Linear(in_features=d_model, out_features=d_k * self.h, bias=False)
        self.value_weights = nn.Linear(in_features=d_model, out_features=d_v * self.h, bias=False)

        self.projection = nn.Linear(in_features=d_v * self.h, out_features=d_model)
    
    def forward(self,x):
        batch_size, seq_len, d_model = x.size() if len(x.size()) == 3 else (1,*x.size())

        # shape: (batch_size, seq_len, d_k * heads)
        queries = self.query_weights(x)
        keys = self.key_weights(x)

        # shape: (batch_size, seq_len, d_v * heads)
        values = self.value_weights(x)

        # first view it (the last dimension: d_k * heads is split in two) then swap seq_len and heads so the last two dimensions work with attention
        # shape: (batch_size, heads, seq_len, d_k)
        queries = queries.view(batch_size, seq_len, self.h, self.d_k).transpose(-3,-2)
        keys = keys.view(batch_size, seq_len, self.h, self.d_k).transpose(-3,-2)
        
        values = values.view(batch_size, seq_len, self.h, self.d_k).transpose(-3,-2)

        # size will be (batch_size, heads, seq_len, d_v)
        results = self.attention(queries, keys, values)
        results = results.transpose(-3,-2)
        results = results.contiguous()
        results = results.view(batch_size, seq_len, self.d_v * self.h)
        results = self.projection(results)
        return results
    



## Testing Attention Outputs

In [9]:
block = Attention(8,8)
block.forward(torch.randn((9,8)),torch.randn((9,8)),torch.randn((9,8)))



tensor([[ 0.0539, -0.0274,  0.0974,  0.4102, -0.2903, -0.3627,  0.0609,  0.0032],
        [ 0.6274,  0.7972,  0.6376,  0.4633,  0.0666, -1.3054,  0.4254, -0.5592],
        [ 0.0543, -0.2193,  0.1177, -0.0346,  0.2917, -0.2215,  0.2813, -0.1132],
        [-0.0985,  0.0177,  0.1603, -0.2896, -0.0992, -0.5926,  0.7382,  0.0370],
        [-0.4235,  0.0989,  0.1493, -0.2611,  0.1266, -0.2285,  0.6303,  0.0851],
        [-0.0308, -0.2437, -1.1513,  0.2134,  1.0226, -0.0414,  0.6555,  1.0193],
        [-0.0182, -0.4552, -0.5246, -0.0089,  1.0122, -0.1046,  0.5873,  0.2833],
        [ 0.1287,  0.1958,  0.3805,  0.4371, -0.2484, -0.6490,  0.1210, -0.2844],
        [-1.0646,  0.3151,  0.1429, -0.5786,  0.5663, -0.1604,  1.2077,  0.2544]])

In [11]:
mh_attention = MultiHeadAttention(8,8,8)

# batch_size, seq_len, d_model
out = mh_attention.forward(torch.randn((2,9,8)))
out.shape

torch.Size([2, 9, 8])

## Normalization and Residual Additions

In transformer models, after the attention computation, the result is added with the input (this allows for better gradients), and normalized.

The normalization step is done using Layer Normalization.

In [ ]:
class LayerNorm(nn.Module):

    def __init__(self, normalized_shape):

        self.normalized_shape = normalized_shape
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))
    
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)

        # gamma gets multiplied pointwise, beta gets added.
        return self.gamma * (x - mean) / std + self.beta
        

